In [2]:
import torch

x = torch.arange(4, dtype=torch.float32, requires_grad=True)

y = torch.dot(x,x)
print(y)

y.sum().backward()
x.grad == 2*x

tensor(14., grad_fn=<DotBackward0>)


tensor([True, True, True, True])

# Jacobian Matrix and Backpropagation

Consider a vector-valued function:

$$
y=f(x)
$$

where:

$$
x\in\mathbb{R}^{n}
$$

is the input vector and:

$$
y\in\mathbb{R}^{m}
$$

is the output vector.

---

## Jacobian Matrix

The derivative of a vector output with respect to a vector input is called the **Jacobian matrix**:

$$
J=\frac{\partial y}{\partial x}
$$

The element at row \(i\) and column \(j\) is:

$$
J_{ij}=\frac{\partial y_i}{\partial x_j}
$$

Therefore:

$$
J=
\begin{bmatrix}
\frac{\partial y_1}{\partial x_1}
&
\frac{\partial y_1}{\partial x_2}
&
\cdots
&
\frac{\partial y_1}{\partial x_n}
\\
\frac{\partial y_2}{\partial x_1}
&
\frac{\partial y_2}{\partial x_2}
&
\cdots
&
\frac{\partial y_2}{\partial x_n}
\\
\vdots
&
\vdots
&
\ddots
&
\vdots
\\
\frac{\partial y_m}{\partial x_1}
&
\frac{\partial y_m}{\partial x_2}
&
\cdots
&
\frac{\partial y_m}{\partial x_n}
\end{bmatrix}
$$

The Jacobian describes how each output element changes when each input element changes.

---

# Gradient of Loss Function

In deep learning, we usually do not need the complete Jacobian matrix.

Instead, we need the gradient of a scalar loss function:

$$
L=L(y)
$$

with respect to the input:

$$
\frac{\partial L}{\partial x}
$$

Using the chain rule:

$$
\frac{\partial L}{\partial x}
=
\frac{\partial L}{\partial y}
\frac{\partial y}{\partial x}
$$

where:

- \( \frac{\partial y}{\partial x} \) is the Jacobian matrix.
- \( \frac{\partial L}{\partial y} \) is the gradient from the loss function.

The multiplication:

$$
\frac{\partial L}{\partial y}
\frac{\partial y}{\partial x}
$$

is called the **vector-Jacobian product (VJP)**.

---

# Why not calculate the full Jacobian?

For large neural networks, the Jacobian can become extremely large.

For example:

$$
x\in\mathbb{R}^{1000000}
$$

and

$$
y\in\mathbb{R}^{1000000}
$$

would produce a Jacobian with:

$$
1000000\times1000000=10^{12}
$$

elements.

Therefore, frameworks such as PyTorch avoid explicitly creating the Jacobian.

Instead, they directly compute:

$$
\frac{\partial L}{\partial y}
\frac{\partial y}{\partial x}
$$

during backpropagation.

---

# Gradient of Loss with Respect to Output

A common misunderstanding is:

$$
\frac{\partial L}{\partial y}
=
[1,1,\dots,1]
$$

This is only true for the special case:

$$
L=\sum_i y_i
$$

because:

$$
\frac{\partial L}{\partial y}
=
\begin{bmatrix}
1\\
1\\
\vdots\\
1
\end{bmatrix}
$$

However, most loss functions do not have this gradient.

For example, Mean Squared Error (MSE):

$$
L=
\frac{1}{2}
\sum_i(y_i-t_i)^2
$$

where \(t\) is the target value.

The gradient is:

$$
\frac{\partial L}{\partial y}
=
y-t
$$

which is generally not a vector of ones.

---

# Backpropagation Summary

The gradient propagated backward through a neural network is:

$$
\frac{\partial L}{\partial x}
=
\frac{\partial L}{\partial y}
\frac{\partial y}{\partial x}
$$

The Jacobian represents the relationship between input and output.

The loss gradient provides the importance of each output.

Their product gives the gradient needed to update model parameters.

In [ ]:
x = torch.tensor([5, 6],dtype=torch.float32, requires_grad=True)
W = torch.tensor([[1, 2],[3, 4]], dtype=torch.float32, requires_grad=True)
y = W @ x
print("Input:",x)
print("Weights:", W)
print("Output:", y)

y_true = torch.tensor([10, 12])

MSE = (1/2 * (y - y_true) ** 2) .sum()

print("MSE:", MSE)

MSE.backward()

print("Gradient of W over x:")
print(W.grad)
print(x.grad)



Input: tensor([5., 6.], requires_grad=True)
Weights: tensor([[1., 2.],
        [3., 4.]], requires_grad=True)
Output: tensor([17., 39.], grad_fn=<MvBackward0>)
MSE: tensor(389., grad_fn=<SumBackward0>)
Gradient of W over x:
tensor([[ 35.,  42.],
        [135., 162.]])
tensor([ 88., 122.])
